In [1]:
import os, sys, pandas as pd, numpy as np, re
sys.path.append(r'C:\Users\Tanushree.Pandey\Downloads\starterRepo\utils')
sys.path.append(r'C:\Users\Tanushree.Pandey\Downloads\starterRepo\connectors')
from math import radians, sin, cos, sqrt, atan2
from sheetHelper import *
from queryHelper import *


from datetime import datetime as dt, date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

In [2]:
append_logs = []
raw_logs = []

In [3]:
API_URL = (
    "https://api.upgrid.in/"
    "api/batteryImmobilize/createLimeRequest"
)

headers = {
    "Content-Type": "application/json"
}

In [4]:
def extract_request_id(resp_text):

    try:
        data = json.loads(resp_text)
    except Exception:
        return None

    keys = [
        "requestId",
        "request_id",
        "limeRequestId"
    ]

    def search(obj):

        if isinstance(obj, dict):

            for k in keys:

                if k in obj and obj[k]:
                    return obj[k]

            for v in obj.values():

                found = search(v)

                if found:
                    return found

        elif isinstance(obj, list):

            for item in obj:

                found = search(item)

                if found:
                    return found

        return None

    return search(data)


def send_command(serial_no, command):

    payload = {
        "serialNo": str(serial_no),
        "command": command
    }

    sent_at = datetime.now(
        pytz.timezone("Asia/Kolkata")
    ).strftime(
        "%Y-%m-%d %H:%M:%S.%f"
    ) + "+00:00"

    try:

        response = requests.post(
            API_URL,
            json=payload,
            headers=headers,
            timeout=20
        )

        status_code = response.status_code

        try:

            result = response.json()

        except Exception:

            result = {}

        success = result.get("success")

        message = result.get("message")

        error_code = None
        detail = None
        description = None

        if success:

            description = (
                result
                .get("data", {})
                .get("message")
            )

            api_execution_status = (
                f"API execution successful: "
                f"{serial_no}"
            )

        else:

            error = result.get(
                "error",
                {}
            )

            error_code = error.get(
                "code"
            )

            detail = error.get(
                "detail"
            )

            description = error.get(
                "description"
            )

            api_execution_status = (
                f"API execution failed: "
                f"{serial_no}"
            )

        request_id = (
            extract_request_id(
                response.text
            )
        )

        return {

            "success": success,

            "status_code":
                status_code,

            "request_id":
                request_id,

            "message":
                message,

            "error_code":
                error_code,

            "detail":
                detail,

            "description":
                description,

            "api_execution_status":
                api_execution_status,

            "response":
                response.text,

            "command":
                command,

            "serialNo":
                str(serial_no),

            "sent_at":
                sent_at

        }

    except Exception as e:

        return {

            "success": False,

            "status_code": None,

            "request_id": None,

            "message": str(e),

            "error_code": None,

            "detail": None,

            "description": str(e),

            "api_execution_status":
                f"API execution failed: {serial_no}",

            "response": str(e),

            "command": command,

            "serialNo": str(serial_no),

            "sent_at": sent_at

        }

In [5]:
gc = pg.authorize(service_file=r'C:\Users\Tanushree.Pandey\Desktop\analytics\analytics.json')  # Update with your credentials
# Load Sheet
SHEET_URL = "https://docs.google.com/spreadsheets/d/12kPyTZb1PILSulIN6uITJwy3uCvg-c8TJtgFd9UEZmw/edit?usp=sharing"  # Update with your actual Google Sheet URL
TAB_NAME = "logs"  # Update with the correct tab name
sh = gc.open_by_url(SHEET_URL)
ws = sh.worksheet_by_title(TAB_NAME)
tracking_df = ws.get_as_df(
    numerize=False
)

In [6]:
tracking_df['chg_rid'] = tracking_df['chg_rid'].astype(str)

In [7]:
tracking_df.tail(5)

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
11810,B582635,E8775156AE0E8D36,TRONTEK,TK-5145-05LX-C-015606,2026-06-29 10:26:41.732011+00:00,2026-06-29,NaN,turned_on,CHGON,DISCHGON,321441_3,321441_4,,
11811,B581061,E7DA5966E9281722,TRONTEK,TK-5145-17KX-C-014357,2026-06-29 10:26:43.959609+00:00,2026-06-29,NaN,turned_on,CHGON,DISCHGON,314835_2,314835_3,26.933151,75.807655
11812,B638077,7EDCEA7977F29049,TRONTEK,TK-5145-20AY-C-019691,2026-06-29 10:26:46.017909+00:00,2026-06-29,NaN,turned_on,CHGON,DISCHGON,309118_5,309118_6,26.93339,75.80768
11813,B959852,3688EFC2F6942439,TRONTEK,TK-5145-20AY-C-019776,2026-06-29 10:26:48.071032+00:00,2026-06-29,NaN,turned_on,CHGON,DISCHGON,300253_4,300253_5,,
11814,B691037,0814619BA7B6B5C6,TRONTEK,TK-5145-29CY-C-027405,2026-06-29 10:26:50.628058+00:00,2026-06-29,NaN,turned_on,CHGON,DISCHGON,389154_5,389154_6,25.29922,82.97022


In [8]:
print(tracking_df[['chg_rid','dischg_rid']].dtypes)

chg_rid       object
dischg_rid    object
dtype: object


In [9]:
print(tracking_df[['chg_rid', 'dischg_rid']].tail())

print(tracking_df['chg_rid'].iloc[0], type(tracking_df['chg_rid'].iloc[0]))
print(tracking_df['dischg_rid'].iloc[0], type(tracking_df['dischg_rid'].iloc[0]))

        chg_rid dischg_rid
11810  321441_3   321441_4
11811  314835_2   314835_3
11812  309118_5   309118_6
11813  300253_4   300253_5
11814  389154_5   389154_6
 <class 'str'>
 <class 'str'>


In [10]:
today_date = (
    pd.Timestamp.now(
        tz="Asia/Kolkata"
    ).date()
)

In [11]:
tracking_df["action_timestamp"] = pd.to_datetime(pd.to_datetime(tracking_df['action_timestamp']).dt.tz_localize(None))
tracking_df.tail()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
11810,B582635,E8775156AE0E8D36,TRONTEK,TK-5145-05LX-C-015606,2026-06-29 10:26:41.732011,2026-06-29,NaN,turned_on,CHGON,DISCHGON,321441_3,321441_4,,
11811,B581061,E7DA5966E9281722,TRONTEK,TK-5145-17KX-C-014357,2026-06-29 10:26:43.959609,2026-06-29,NaN,turned_on,CHGON,DISCHGON,314835_2,314835_3,26.933151,75.807655
11812,B638077,7EDCEA7977F29049,TRONTEK,TK-5145-20AY-C-019691,2026-06-29 10:26:46.017909,2026-06-29,NaN,turned_on,CHGON,DISCHGON,309118_5,309118_6,26.93339,75.80768
11813,B959852,3688EFC2F6942439,TRONTEK,TK-5145-20AY-C-019776,2026-06-29 10:26:48.071032,2026-06-29,NaN,turned_on,CHGON,DISCHGON,300253_4,300253_5,,
11814,B691037,0814619BA7B6B5C6,TRONTEK,TK-5145-29CY-C-027405,2026-06-29 10:26:50.628058,2026-06-29,NaN,turned_on,CHGON,DISCHGON,389154_5,389154_6,25.29922,82.97022


In [12]:
tracking_df.tail()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
11810,B582635,E8775156AE0E8D36,TRONTEK,TK-5145-05LX-C-015606,2026-06-29 10:26:41.732011,2026-06-29,NaN,turned_on,CHGON,DISCHGON,321441_3,321441_4,,
11811,B581061,E7DA5966E9281722,TRONTEK,TK-5145-17KX-C-014357,2026-06-29 10:26:43.959609,2026-06-29,NaN,turned_on,CHGON,DISCHGON,314835_2,314835_3,26.933151,75.807655
11812,B638077,7EDCEA7977F29049,TRONTEK,TK-5145-20AY-C-019691,2026-06-29 10:26:46.017909,2026-06-29,NaN,turned_on,CHGON,DISCHGON,309118_5,309118_6,26.93339,75.80768
11813,B959852,3688EFC2F6942439,TRONTEK,TK-5145-20AY-C-019776,2026-06-29 10:26:48.071032,2026-06-29,NaN,turned_on,CHGON,DISCHGON,300253_4,300253_5,,
11814,B691037,0814619BA7B6B5C6,TRONTEK,TK-5145-29CY-C-027405,2026-06-29 10:26:50.628058,2026-06-29,NaN,turned_on,CHGON,DISCHGON,389154_5,389154_6,25.29922,82.97022


In [13]:
tracking_df["action_date"] = (
    tracking_df["action_timestamp"]
    .dt.date
)

In [14]:
today_ist = pd.Timestamp.now()

window_start = today_ist - pd.Timedelta(days=3)

print(window_start)
print(today_ist)

2026-06-26 10:28:48.504102
2026-06-29 10:28:48.504102


In [15]:
## all last 3 days tracking

last_3_day_tracking = tracking_df[
    (
        tracking_df["action_timestamp"]
        >= window_start
    )
    &
    (
        tracking_df["action_timestamp"]
        <= today_ist
    )
].copy()

In [16]:
last_3_day_tracking.tail()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
11810,B582635,E8775156AE0E8D36,TRONTEK,TK-5145-05LX-C-015606,2026-06-29 10:26:41.732011,2026-06-29,NaN,turned_on,CHGON,DISCHGON,321441_3,321441_4,,
11811,B581061,E7DA5966E9281722,TRONTEK,TK-5145-17KX-C-014357,2026-06-29 10:26:43.959609,2026-06-29,NaN,turned_on,CHGON,DISCHGON,314835_2,314835_3,26.933151,75.807655
11812,B638077,7EDCEA7977F29049,TRONTEK,TK-5145-20AY-C-019691,2026-06-29 10:26:46.017909,2026-06-29,NaN,turned_on,CHGON,DISCHGON,309118_5,309118_6,26.93339,75.80768
11813,B959852,3688EFC2F6942439,TRONTEK,TK-5145-20AY-C-019776,2026-06-29 10:26:48.071032,2026-06-29,NaN,turned_on,CHGON,DISCHGON,300253_4,300253_5,,
11814,B691037,0814619BA7B6B5C6,TRONTEK,TK-5145-29CY-C-027405,2026-06-29 10:26:50.628058,2026-06-29,NaN,turned_on,CHGON,DISCHGON,389154_5,389154_6,25.29922,82.97022


In [17]:

print(last_3_day_tracking["action_date"].max())
print(last_3_day_tracking["action_date"].min())
len(last_3_day_tracking)

2026-06-29
2026-06-26


262

In [18]:
## only the turn off command of last 3 days
last_3_day_tracking=last_3_day_tracking[last_3_day_tracking['action']!='turned_on']
last_3_day_tracking

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
11579,B465368,042244F406B64EF6,STEFEN ELECTRIC,G24-49848,2026-06-26 12:10:48.101747,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,253854_4,28.662706,77.18083
11580,B680698,14AB1FB557983CEA,CYGNI ENERGY,E2W5140BS4024960,2026-06-26 12:10:49.668328,2026-06-26,Charging,turned_off,NaN,DISCHGOFF,NaN,288397_249,28.621607,77.329285
11581,B599513,1A508CC5BA024DF5,INVERTED,INDMXH5140E0898,2026-06-26 12:10:50.672926,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,259262_1,28.616825,77.07752
11582,B509290,1B33028AFEC4B409,BATTRIXX,B112003240_122029,2026-06-26 12:10:51.689172,2026-06-26,Discharging,turned_off,CHGOFF,NaN,265884_3,NaN,26.954678,75.865555
11583,B849755,2A11A679A461B6E5,STEFEN ELECTRIC,H24-52916,2026-06-26 12:10:52.891339,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,259596_1,28.588556,77.364395
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11775,B769624,30522C43A6157217,INVERTED,INCBYH5145E1320,2026-06-26 16:10:40.896202,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,461293_8,,
11776,B902526,2D203487FE1C6F8A,BATTRIXX,B112003240_121494,2026-06-26 16:10:41.912223,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,262006_5,,
11777,B607994,605F0EDEE94D048B,STEFEN ELECTRIC,B24-33490,2026-06-26 16:10:53.401240,2026-06-26,Discharging,turned_off,CHGOFF,NaN,169610_25,NaN,,
11778,B709679,331FC418E7902E2E,IPOWER,HHL48452507011864,2026-06-26 16:10:54.414728,2026-06-26,Charging,turned_off,CHGOFF,NaN,446417_3,NaN,,


In [19]:
from datetime import datetime as dt
already_on= tracking_df.copy()
already_on = (
    already_on
    .sort_values(
        ["batteryId", "action_timestamp"]
    )
    .groupby("batteryId", as_index=False)
    .tail(1)
    .copy()
)
already_on=already_on[already_on['action']=='turned_on']

already_on=already_on[already_on['action_date']!=dt.today().date()]
already_on

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
5093,B1000202,246F4B3F82B5A214,BATTRIXX,B112004420_131931,2026-06-03 08:11:10.491902,2026-06-03,,turned_on,,,,,,
11270,B1000852,80D52CC004987C7A,BATTRIXX,B112004450_127601,2026-06-25 22:07:50.307864,2026-06-25,NaN,turned_on,CHGON,DISCHGON,652638_12,652638_13,,
10466,B1000872,20327703C839920D,BATTRIXX,B112004450_127580,2026-06-24 22:07:45.534861,2026-06-24,NaN,turned_on,CHGON,DISCHGON,643340_2,643340_3,,
11284,B1006072,B80971E0D89C3B21,STEFEN ELECTRIC,G24-46217,2026-06-25 22:08:20.705802,2026-06-25,NaN,turned_on,CHGON,DISCHGON,230821_16,230821_17,28.69154,77.32889
7210,B1009852,A2A06BEDAE222E89,INVERTED,INDMWE514001559,2026-06-08 20:11:01.835321,2026-06-08,,turned_on,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11698,B998600,67F999FA7150EAF9,INVERTED,INCBZF5145E0261,2026-06-26 16:08:18.366353,2026-06-26,NaN,turned_on,CHGON,DISCHGON,669872_6,669872_7,26.933382,75.80772
11303,B998744,67A85D4C8EE6F3B4,INVERTED,INCBZF5145E0378,2026-06-25 22:09:00.812158,2026-06-25,NaN,turned_on,CHGON,DISCHGON,600654_2,600654_3,26.875656,80.922935
11699,B998828,54945029FB58401D,INVERTED,INCBZF5145E0458,2026-06-26 16:08:20.465774,2026-06-26,NaN,turned_on,CHGON,DISCHGON,667067_8,667067_9,26.933294,75.807686
10489,B998831,BBDF3E55A07E68BB,INVERTED,INCBZF5145E0461,2026-06-24 22:08:36.311968,2026-06-24,NaN,turned_on,CHGON,DISCHGON,669823_8,669823_9,,


In [20]:
## last turn off command for the last 3 days

latest_tracking_last3 = (last_3_day_tracking.sort_values(["serialNo", "action_timestamp"]).groupby("serialNo", as_index=False).tail(1).copy())
len(latest_tracking_last3)

156

## to be discussed
the code earlier was shutting mosfet on once. because if last is turn on, then exclude that battery from mosfet actioning

In [351]:
# latest_tracking_last3 = latest_tracking_last3[
#     latest_tracking_last3["action"] != "turned_on"
# ].copy()
# print(len(latest_tracking_last3))

In [21]:
batteries = prodFetch(f""" select id as batteryId, serialNo, iotDeviceNo from batteries b where isBaas=0 """)

txnLogs = prodFetch(f"""
select t.partnerId,t.driverId, t.batteriesIssued, t.batteriesReceived,t.createdAt, t.updatedAt, 
t.status,t.id as txnId from transactions t
where createdAt >= CURRENT_DATE() - interval 3 day 
and t.deletedAt is  null """)

txnLogs['createdAt'] = txnLogs['createdAt']+pd.Timedelta(minutes=330)
txnLogs['updatedAt'] = txnLogs['updatedAt']+pd.Timedelta(minutes=330)


txnLogs['date'] = txnLogs['createdAt'].dt.date

import ast
txnLogs['batteriesReceived'] = txnLogs['batteriesReceived'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])
txnLogs['batteriesIssued'] = txnLogs['batteriesIssued'].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else [])

txnLogs['batteryRc1'] = txnLogs['batteriesReceived'].apply(lambda x: x[0] if len(x) > 0 else None)
txnLogs['batteryRc2'] = txnLogs['batteriesReceived'].apply(lambda x: x[1] if len(x) > 1 else None)
txnLogs['batteryIs1'] = txnLogs['batteriesIssued'].apply(lambda x: x[0] if len(x) > 0 else None)
txnLogs['batteryIs2'] = txnLogs['batteriesIssued'].apply(lambda x: x[1] if len(x) > 1 else None)


txnLogs = txnLogs.melt(
    id_vars=['partnerId', 'driverId', 'createdAt','updatedAt','date'],
    value_vars=['batteryRc1', 'batteryRc2', 'batteryIs1', 'batteryIs2'],
    var_name='batteryType',
    value_name='batteryId'
)
txnLogs2 = txnLogs.merge(batteries,on='batteryId',how='left')
txnLogs2.sort_values(by='createdAt',inplace=True)
txnLogs2=txnLogs2[txnLogs2['serialNo'].isna()==False]
txnLogs2

,partnerId,driverId,createdAt,updatedAt,date,batteryType,batteryId,serialNo,iotDeviceNo
0,P5055,D227784,2026-06-26 05:30:02,2026-06-26 05:31:21,2026-06-26,batteryRc1,B580397,TK-5145-26JX-C-013743,378776E121E8EB57
190841,P5055,D227784,2026-06-26 05:30:02,2026-06-26 05:31:21,2026-06-26,batteryRc2,B535846,TK-5145-26HX-C-006455,A6E11CC03C2CC2A1
381682,P5055,D227784,2026-06-26 05:30:02,2026-06-26 05:31:21,2026-06-26,batteryIs1,B652517,B112004180_121355,AE1246987012C45D
572523,P5055,D227784,2026-06-26 05:30:02,2026-06-26 05:31:21,2026-06-26,batteryIs2,B694453,B112004180_130888,40010D5641E8CC2E
1,P4722,D228355,2026-06-26 05:30:11,2026-06-26 05:31:39,2026-06-26,batteryRc1,B768303,HHL48452508013734,A2042798104966E5
...,...,...,...,...,...,...,...,...,...
190839,P5486,D271136,2026-06-29 10:28:45,2026-06-29 10:29:09,2026-06-29,batteryRc1,B999548,B112004420_131809,284AA058ED74CC17
381681,P1206,D72043,2026-06-29 10:28:49,2026-06-29 10:29:14,2026-06-29,batteryRc2,B562657,I24-57648,9010F93B2C8EEB7A
190840,P1206,D72043,2026-06-29 10:28:49,2026-06-29 10:29:14,2026-06-29,batteryRc1,B1010420,L24-68154,FE695E7F791CFC07
572522,P1206,D72043,2026-06-29 10:28:49,2026-06-29 10:29:14,2026-06-29,batteryIs1,B385084,F23-15123,CA734A74976CDEEC


In [22]:
import ast
import json

# =========================================================
# COPY
# =========================================================

txn_df = txnLogs2.copy()

In [23]:
merged = latest_tracking_last3.merge(

    txn_df[[
        "createdAt",
        "serialNo"
    ]],

    on="serialNo",
    how="left"
)

merged.head()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,createdAt
0,B656899,D26F7FA0B350D2CC,Epitome Sustainable Energy Private Limited.,10703250410554,2026-06-26 12:11:19.849825,2026-06-26,Discharging,turned_off,CHGOFF,NaN,373260_2,NaN,26.927105,75.843666,NaT
1,B693935,CECE7E557DC5EB0B,Epitome Sustainable Energy Private Limited.,10704250411543,2026-06-26 14:11:25.995906,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,376061_2,26.801466,75.81232,NaT
2,B697006,DAAD32FDFA5F1D3C,Epitome Sustainable Energy Private Limited.,10704250411779,2026-06-26 16:09:45.547956,2026-06-26,Discharging,turned_off,CHGOFF,NaN,375982_1,NaN,26.954655,75.86558,NaT
3,B754100,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,10707250414530,2026-06-26 16:09:51.387218,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,443316_2,26.954672,75.86559,2026-06-28 11:38:47
4,B754100,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,10707250414530,2026-06-26 16:09:51.387218,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,443316_2,26.954672,75.86559,2026-06-28 14:16:54


In [24]:
final = merged[
    merged["createdAt"]
    >
    merged["action_timestamp"]
].copy()

In [25]:
final = final.sort_values(
    "createdAt"
)

len(final)

95

In [26]:
final

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,createdAt
201,B489566,CBCD6C3624E34A1E,TRONTEK,TK-5145-06FX-C-000810,2026-06-26 12:11:18.759955,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,220547_1,28.690557,77.32913,2026-06-26 13:50:18
32,B650983,4F9ADC25817C8581,BATTRIXX,B112004180_127085,2026-06-26 12:11:58.550443,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,358560_2,26.923298,75.8479,2026-06-26 14:25:16
89,B896625,82FC0791BA9BC413,STEFEN ELECTRIC,G24-45889,2026-06-26 12:11:07.374155,2026-06-26,Discharging,turned_off,CHGOFF,NaN,228543_1,NaN,28.981495,77.689735,2026-06-26 14:37:14
71,B562042,66713D7026E5AF85,STEFEN ELECTRIC,C24-34593,2026-06-26 12:11:04.314091,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,171758_1,28.981815,77.68985,2026-06-26 14:59:43
96,B485074,5490B18B43C10889,STEFEN ELECTRIC,H24-53678,2026-06-26 12:11:00.337424,2026-06-26,Discharging,turned_off,CHGOFF,NaN,266579_1,NaN,28.981602,77.68953,2026-06-26 14:59:43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,B638077,7EDCEA7977F29049,TRONTEK,TK-5145-20AY-C-019691,2026-06-26 14:11:17.010784,2026-06-26,Charging,turned_off,NaN,DISCHGOFF,NaN,309118_4,26.93339,75.80768,2026-06-28 22:20:48
179,B977995,707D751973124CF4,STEFEN ELECTRIC,L24-67827,2026-06-26 14:11:14.466447,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,308941_1,26.933447,75.80764,2026-06-28 22:22:51
10,B798908,5A1A75AA39F56CB9,Epitome Sustainable Energy Private Limited.,10708250415634,2026-06-26 16:09:31.474498,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,471411_2,26.954609,75.86561,2026-06-28 22:48:23
6,B754100,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,10707250414530,2026-06-26 16:09:51.387218,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,443316_2,26.954672,75.86559,2026-06-28 22:48:23


In [27]:
final = (
    final
    .sort_values(["serialNo", "createdAt"])
    .drop_duplicates(
        subset=["serialNo"],
        keep="first"
    )
    .reset_index(drop=True)
)
len(final)

40

In [28]:
final=final[~final['serialNo'].isin(already_on['serialNo'])]
final

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,createdAt
0,B754100,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,10707250414530,2026-06-26 16:09:51.387218,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,443316_2,26.954672,75.86559,2026-06-28 11:38:47
1,B798908,5A1A75AA39F56CB9,Epitome Sustainable Energy Private Limited.,10708250415634,2026-06-26 16:09:31.474498,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,471411_2,26.954609,75.86561,2026-06-28 11:38:47
2,B509392,F2E9CE317CE848B0,BATTRIXX,B112003240_121738,2026-06-26 12:12:00.868748,2026-06-26,Charging,turned_off,CHGOFF,NaN,258439_2,NaN,26.927265,75.8436,2026-06-26 21:32:09
3,B650389,BB26662D3C9E370B,BATTRIXX,B112004180_124872,2026-06-26 14:12:06.092085,2026-06-26,Charging,turned_off,NaN,DISCHGOFF,NaN,354012_2,28.610195,77.349144,2026-06-27 19:40:42
5,B712210,D43DD5DFD5F824F1,BATTRIXX,B112004420_124019,2026-06-26 14:11:57.739421,2026-06-26,Charging,turned_off,NaN,DISCHGOFF,NaN,424857_4,,,2026-06-28 21:33:13
6,B761639,3B3446290EAA90C8,BATTRIXX,B112004420_127694,2026-06-26 12:11:48.828194,2026-06-26,Discharging,turned_off,CHGOFF,NaN,451153_2,NaN,,,2026-06-26 19:07:25
7,B763027,9C792A6EC4321CD8,BATTRIXX,B112004420_128622,2026-06-26 14:11:56.703708,2026-06-26,Charging,turned_off,CHGOFF,NaN,451289_5,NaN,,,2026-06-28 21:33:13
8,B564008,8F6B93762B9DA90F,STEFEN ELECTRIC,B23-01194,2026-06-26 16:09:38.237240,2026-06-26,Discharging,turned_off,CHGOFF,NaN,122945_3,NaN,26.877157,80.91448,2026-06-28 14:22:58
9,B936581,41A8CA96D540B064,STEFEN ELECTRIC,C24-32765,2026-06-26 14:11:09.249443,2026-06-26,Charging,turned_off,NaN,DISCHGOFF,NaN,169539_2,27.563803,77.68424,2026-06-26 17:33:00
11,B976076,4702934D96DD56F7,STEFEN ELECTRIC,C24-35230,2026-06-26 16:09:29.455332,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,169839_4,28.608744,77.31839,2026-06-27 14:06:49


In [29]:
from datetime import datetime
import pytz

current_time = datetime.now(
    pytz.timezone("Asia/Kolkata")
)

current_time = current_time.strftime(
    "%Y-%m-%d %H:%M:%S.%f"
) + "+00:00"

print(current_time)

2026-06-29 10:29:52.513149+00:00


In [30]:
final.head()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,createdAt
0,B754100,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,10707250414530,2026-06-26 16:09:51.387218,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,443316_2,26.954672,75.86559,2026-06-28 11:38:47
1,B798908,5A1A75AA39F56CB9,Epitome Sustainable Energy Private Limited.,10708250415634,2026-06-26 16:09:31.474498,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,471411_2,26.954609,75.86561,2026-06-28 11:38:47
2,B509392,F2E9CE317CE848B0,BATTRIXX,B112003240_121738,2026-06-26 12:12:00.868748,2026-06-26,Charging,turned_off,CHGOFF,NaN,258439_2,NaN,26.927265,75.8436,2026-06-26 21:32:09
3,B650389,BB26662D3C9E370B,BATTRIXX,B112004180_124872,2026-06-26 14:12:06.092085,2026-06-26,Charging,turned_off,NaN,DISCHGOFF,NaN,354012_2,28.610195,77.349144,2026-06-27 19:40:42
5,B712210,D43DD5DFD5F824F1,BATTRIXX,B112004420_124019,2026-06-26 14:11:57.739421,2026-06-26,Charging,turned_off,NaN,DISCHGOFF,NaN,424857_4,,,2026-06-28 21:33:13


In [31]:
turn_on = final.copy()

turn_on = turn_on[
    [
        "batteryId",
        "deviceId","manufacturerName","serialNo",
        "action_timestamp","lat","lon"
    ]
].copy()



turn_on.head()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,lat,lon
0,B754100,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,10707250414530,2026-06-26 16:09:51.387218,26.954672,75.86559
1,B798908,5A1A75AA39F56CB9,Epitome Sustainable Energy Private Limited.,10708250415634,2026-06-26 16:09:31.474498,26.954609,75.86561
2,B509392,F2E9CE317CE848B0,BATTRIXX,B112003240_121738,2026-06-26 12:12:00.868748,26.927265,75.8436
3,B650389,BB26662D3C9E370B,BATTRIXX,B112004180_124872,2026-06-26 14:12:06.092085,28.610195,77.349144
5,B712210,D43DD5DFD5F824F1,BATTRIXX,B112004420_124019,2026-06-26 14:11:57.739421,,


In [32]:
from time import time, sleep, perf_counter

## to be discussed
partial failure

In [33]:
for idx, row in turn_on.iterrows():

    battery_id = row["batteryId"]
    serial_no = row["serialNo"]
    device_id = row["deviceId"]
    manufacturer_name = row["manufacturerName"]
    lat=row["lat"]
    lon=row["lon"]


    current_time = datetime.now(
        pytz.timezone("Asia/Kolkata")
    ).strftime(
        "%Y-%m-%d %H:%M:%S.%f"
    ) + "+00:00"

    print(f"\nPROCESSING -> {battery_id}")
   
    print("  SENDING TURN ON")

    # =====================================
    # CHGON
    # =====================================

    chg_on_resp = send_command(
        serial_no,
        "CHGON"
    )

 

    # =====================================
    # DISCHGON
    # =====================================

    dischg_on_resp = send_command(
        serial_no,
        "DISCHGON"
    )



    chg_success = chg_on_resp["success"]
    dischg_success = dischg_on_resp["success"]

    # =====================================
    # SUCCESS ROW
    # =====================================

    if chg_success or dischg_success:

        success_resp = (
            chg_on_resp
            if chg_success
            else dischg_on_resp
        )

        append_logs.append({

            "batteryId": battery_id,
            "serialNo": serial_no,
            "deviceId": device_id,
            "manufacturerName": manufacturer_name,
            "lat":lat,
            "lon":lon,

            "action_timestamp": current_time,
            "action": "turned_on",

            "CHG CMD": "CHGON",
            "DICHG CMD": "DISCHGON",

            "chg_rid": chg_on_resp["request_id"],
            "dischg_rid": dischg_on_resp["request_id"]

        })
    # =====================================
    # FAILURE ROW
    # =====================================

    if (
        not chg_success
        or
        not dischg_success
    ):

        failed_resp = (
            chg_on_resp
            if not chg_success
            else dischg_on_resp
        )

        raw_logs.append({

            "batteryId": battery_id,
            "serialNo": serial_no,
            "deviceId": device_id,
            "manufacturerName": manufacturer_name,
            "lat":lat,
            "lon":lon,

            "action_timestamp": current_time,
            "action": "turn_on_failed",

            "CHG CMD": "CHGON",
            "DICHG CMD": "DISCHGON",

            "chg_rid": chg_on_resp["request_id"],
            "dischg_rid": dischg_on_resp["request_id"]

        })
        print(
            "  TURN ON PARTIAL or FULL FAILED"
        )


PROCESSING -> B754100
  SENDING TURN ON

PROCESSING -> B798908
  SENDING TURN ON

PROCESSING -> B509392
  SENDING TURN ON

PROCESSING -> B650389
  SENDING TURN ON

PROCESSING -> B712210
  SENDING TURN ON

PROCESSING -> B761639
  SENDING TURN ON

PROCESSING -> B763027
  SENDING TURN ON

PROCESSING -> B564008
  SENDING TURN ON

PROCESSING -> B936581
  SENDING TURN ON

PROCESSING -> B976076
  SENDING TURN ON

PROCESSING -> B989688
  SENDING TURN ON

PROCESSING -> B951929
  SENDING TURN ON

PROCESSING -> B465368
  SENDING TURN ON

PROCESSING -> B702716
  SENDING TURN ON

PROCESSING -> B709240
  SENDING TURN ON

PROCESSING -> B616321
  SENDING TURN ON

PROCESSING -> B980296
  SENDING TURN ON

PROCESSING -> B562523
  SENDING TURN ON

PROCESSING -> B770300
  SENDING TURN ON

PROCESSING -> B773437
  SENDING TURN ON

PROCESSING -> B995599
  SENDING TURN ON

PROCESSING -> B997452
  SENDING TURN ON

PROCESSING -> B619571
  SENDING TURN ON

PROCESSING -> B566886
  SENDING TURN ON

PROCESSING -> B

In [34]:
append_logs_df = pd.DataFrame(
    append_logs
)

In [35]:
append_logs_df

,batteryId,serialNo,deviceId,manufacturerName,lat,lon,action_timestamp,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid
0,B754100,10707250414530,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,26.954672,75.86559,2026-06-29 10:55:21.291031+00:00,turned_on,CHGON,DISCHGON,443316_5,443316_6
1,B798908,10708250415634,5A1A75AA39F56CB9,Epitome Sustainable Energy Private Limited.,26.954609,75.86561,2026-06-29 10:55:24.661654+00:00,turned_on,CHGON,DISCHGON,471411_5,471411_6
2,B509392,B112003240_121738,F2E9CE317CE848B0,BATTRIXX,26.927265,75.8436,2026-06-29 10:55:26.898993+00:00,turned_on,CHGON,DISCHGON,258439_5,258439_6
3,B650389,B112004180_124872,BB26662D3C9E370B,BATTRIXX,28.610195,77.349144,2026-06-29 10:55:29.274374+00:00,turned_on,CHGON,DISCHGON,354012_5,354012_6
4,B712210,B112004420_124019,D43DD5DFD5F824F1,BATTRIXX,,,2026-06-29 10:55:33.627797+00:00,turned_on,CHGON,DISCHGON,424857_7,424857_8
5,B761639,B112004420_127694,3B3446290EAA90C8,BATTRIXX,,,2026-06-29 10:55:35.804812+00:00,turned_on,CHGON,DISCHGON,451153_5,451153_6
6,B763027,B112004420_128622,9C792A6EC4321CD8,BATTRIXX,,,2026-06-29 10:55:38.111700+00:00,turned_on,CHGON,DISCHGON,451289_8,451289_9
7,B564008,B23-01194,8F6B93762B9DA90F,STEFEN ELECTRIC,26.877157,80.91448,2026-06-29 10:55:40.386919+00:00,turned_on,CHGON,DISCHGON,122945_6,122945_7
8,B936581,C24-32765,41A8CA96D540B064,STEFEN ELECTRIC,27.563803,77.68424,2026-06-29 10:55:42.648591+00:00,turned_on,CHGON,DISCHGON,169539_5,169539_6
9,B976076,C24-35230,4702934D96DD56F7,STEFEN ELECTRIC,28.608744,77.31839,2026-06-29 10:55:44.922903+00:00,turned_on,CHGON,DISCHGON,169839_7,169839_8


In [36]:
append_logs_df["action_timestamp"] = pd.to_datetime(
    append_logs_df["action_timestamp"]
)

append_logs_df["action_date"] = (
    append_logs_df["action_timestamp"]
    .dt.date
)

In [37]:
append_logs_df["state"]=np.nan

In [38]:
append_logs_df=append_logs_df[["batteryId","deviceId","manufacturerName","serialNo","action_timestamp","action_date","state","action","CHG CMD","DICHG CMD","chg_rid","dischg_rid","lat","lon"]]
append_logs_df.head(2)

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
0,B754100,E1C56D2DAE73D7E0,Epitome Sustainable Energy Private Limited.,10707250414530,2026-06-29 10:55:21.291031+00:00,2026-06-29,NaN,turned_on,CHGON,DISCHGON,443316_5,443316_6,26.954672,75.86559
1,B798908,5A1A75AA39F56CB9,Epitome Sustainable Energy Private Limited.,10708250415634,2026-06-29 10:55:24.661654+00:00,2026-06-29,NaN,turned_on,CHGON,DISCHGON,471411_5,471411_6,26.954609,75.86561


In [40]:
custom_append(
        "https://docs.google.com/spreadsheets/d/12kPyTZb1PILSulIN6uITJwy3uCvg-c8TJtgFd9UEZmw/edit?gid=1581823282#gid=1581823282",
        "logs",append_logs_df
    )

Appended successfully.


In [41]:
today_filtered_batteries=read("https://docs.google.com/spreadsheets/d/12kPyTZb1PILSulIN6uITJwy3uCvg-c8TJtgFd9UEZmw/edit?gid=0#gid=0","Today_filtered_batteries")
print(len(today_filtered_batteries))

1099


In [42]:
serial_list = today_filtered_batteries["serialNo"].dropna().unique().tolist()

In [43]:
serial_list_sql = ",".join(
    f"'{x}'"
    for x in today_filtered_batteries["serialNo"].dropna().unique()
)


In [44]:
today_txn = txnLogs2[txnLogs2['date']==dt.today().date()]
today_txn

,partnerId,driverId,createdAt,updatedAt,date,batteryType,batteryId,serialNo,iotDeviceNo
564025,P4348,D164702,2026-06-29 00:00:06,2026-06-29 00:01:15,2026-06-29,batteryIs1,B533787,TK-5145-12HX-C-004713,9B2934C8BBBA1518
373184,P4348,D164702,2026-06-29 00:00:06,2026-06-29 00:01:15,2026-06-29,batteryRc2,B750371,TK-5145-24GY-C-030856,1248E8FE39998B41
182343,P4348,D164702,2026-06-29 00:00:06,2026-06-29 00:01:15,2026-06-29,batteryRc1,B942818,TK-5145-01IY-C-034442,9E24FE559BB38F83
182344,P0810,D22665,2026-06-29 00:00:08,2026-06-29 00:02:56,2026-06-29,batteryRc1,B731251,H23-21069,6F8C655A71ABDED3
754867,P0810,D22665,2026-06-29 00:00:08,2026-06-29 00:02:56,2026-06-29,batteryIs2,B604884,INDMWI514000065,DEB72CCD471E7BB2
...,...,...,...,...,...,...,...,...,...
190839,P5486,D271136,2026-06-29 10:28:45,2026-06-29 10:29:09,2026-06-29,batteryRc1,B999548,B112004420_131809,284AA058ED74CC17
381681,P1206,D72043,2026-06-29 10:28:49,2026-06-29 10:29:14,2026-06-29,batteryRc2,B562657,I24-57648,9010F93B2C8EEB7A
190840,P1206,D72043,2026-06-29 10:28:49,2026-06-29 10:29:14,2026-06-29,batteryRc1,B1010420,L24-68154,FE695E7F791CFC07
572522,P1206,D72043,2026-06-29 10:28:49,2026-06-29 10:29:14,2026-06-29,batteryIs1,B385084,F23-15123,CA734A74976CDEEC


In [45]:
remaining_batteries = today_filtered_batteries[
    ~today_filtered_batteries["serialNo"].isin(
        today_txn["serialNo"]
    )
].copy()
print(len(remaining_batteries))

1096


In [46]:
tracking_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11815 entries, 0 to 11814
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   batteryId         11815 non-null  object        
 1   deviceId          11815 non-null  object        
 2   manufacturerName  11815 non-null  object        
 3   serialNo          11815 non-null  object        
 4   action_timestamp  11815 non-null  datetime64[ns]
 5   action_date       11815 non-null  object        
 6   state             11815 non-null  object        
 7   action            11815 non-null  object        
 8   CHG CMD           11815 non-null  object        
 9   DICHG CMD         11815 non-null  object        
 10  chg_rid           11815 non-null  object        
 11  dischg_rid        11815 non-null  object        
 12  lat               11815 non-null  object        
 13  lon               11815 non-null  object        
dtypes: datetime64[ns](1), 

In [47]:
tracking_df = tracking_df[
    tracking_df["action"] != "turned_on"
].copy()
print(len(tracking_df))

9776


In [48]:
tracking_df["action_date"]= pd.to_datetime(tracking_df["action_date"]).dt.date

In [49]:
tracking_df

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
0,B605385,015F0F4C65AAFBBC,STEFEN ELECTRIC,G22-39732,2026-05-21 11:29:43.289487,2026-05-21,,turned_off,,,,,,
1,B399907,D928BD6505A2A461,INVERTED,INDMXD5140E0048,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
2,B405455,C5FBD9AE73EC1B17,LIVGUARD,MD4AIGDNAB06921,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
3,B419006,AC37695380E1959D,LIVGUARD,MD4AIVENBB03819,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
4,B427128,F2453474865D6473,LIVGUARD,MD4AI9GNBB01474,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11775,B769624,30522C43A6157217,INVERTED,INCBYH5145E1320,2026-06-26 16:10:40.896202,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,461293_8,,
11776,B902526,2D203487FE1C6F8A,BATTRIXX,B112003240_121494,2026-06-26 16:10:41.912223,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,262006_5,,
11777,B607994,605F0EDEE94D048B,STEFEN ELECTRIC,B24-33490,2026-06-26 16:10:53.401240,2026-06-26,Discharging,turned_off,CHGOFF,NaN,169610_25,NaN,,
11778,B709679,331FC418E7902E2E,IPOWER,HHL48452507011864,2026-06-26 16:10:54.414728,2026-06-26,Charging,turned_off,CHGOFF,NaN,446417_3,NaN,,


In [50]:
already_actioned=tracking_df[(tracking_df['action_date']>=dt.today().date() - pd.Timedelta(days=2)) & (tracking_df['action']=='turned_off')]
print(len(already_actioned))

0


In [51]:
already_actioned.tail()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon


In [52]:
already_actioned=already_actioned[already_actioned['serialNo'].isin(today_filtered_batteries['serialNo'].values)]

In [53]:
already_actioned

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon


In [54]:
print(already_actioned["action_date"].max())
print(already_actioned["action_date"].min())

nan
nan


In [55]:
remaining_batteries = remaining_batteries[
    ~remaining_batteries["serialNo"].isin(
        already_actioned["serialNo"]
    )
].copy()
print(len(remaining_batteries))

1096


In [56]:
already_actioned

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon


In [57]:
already_actioned["action_timestamp"] = pd.to_datetime(already_actioned["action_timestamp"]).dt.tz_localize(None)

already_actioned=already_actioned.sort_values(by='action_timestamp')
already_actioned.drop_duplicates(subset='deviceId',keep='last',inplace=True)
already_actioned

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon


In [58]:
already_actioned[already_actioned['serialNo']=='B1_51105_B_121703']

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon


In [59]:
already_actioned["completed_24hr"] = (
    pd.Timestamp.now()
    - already_actioned["action_timestamp"]
) >= pd.Timedelta(hours=24)

In [60]:
already_actioned[already_actioned['serialNo']=='B1_51105_B_121703']

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,completed_24hr


In [61]:
completed_hr=already_actioned[already_actioned["completed_24hr"]==True]
print(len(completed_hr))

0


In [62]:
completed_hr.tail()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,completed_24hr


In [63]:
completed_hr['chg_rid'] = completed_hr['chg_rid'].astype(str)
completed_hr['dischg_rid'] = completed_hr['dischg_rid'].astype(str)


In [64]:
request_ids = (
    completed_hr[
        ["chg_rid", "dischg_rid"]
    ]
    .stack()
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

print(request_ids)

[]


In [78]:
completed_hr.tail()

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon,completed_24hr


In [77]:
request_ids_sql = ",".join(
    f"'{x}'"
    for x in request_ids
)

request_ids_sql

''

In [67]:
cmd_rows = []

for _, row in completed_hr.iterrows():

    base = {
        "batteryId": row["batteryId"],
        "deviceId": row["deviceId"],
        "action_timestamp": row["action_timestamp"],
        "action_date": row["action_date"],
        "state": row["state"],
        "action": row["action"],
        "lat":row['lat'],
        "lon":row['lon']
    }

    if pd.notna(row.get("chg_rid")):
        cmd_rows.append({
            **base,
            "command_type": "CHG",
            "command_sent": row.get("CHG CMD"),
            "requestId": row["chg_rid"]
        })

    if pd.notna(row.get("dischg_rid")):
        cmd_rows.append({
            **base,
            "command_type": "DISCHG",
            "command_sent": row.get("DICHG CMD"),
            "requestId": row["dischg_rid"]
        })

command_hr_long = pd.DataFrame(cmd_rows)

In [68]:
print(len(command_hr_long))

0


In [70]:
if len(command_hr_long)>0:
    command_hr_long.sort_values(by='action_timestamp',inplace=True)
    command_hr_long

In [82]:
if len(command_hr_long)>0:

    command_hr_long = command_hr_long[
        command_hr_long["requestId"].notna()
    ].copy()

    command_hr_long = command_hr_long[
        command_hr_long["requestId"].astype(str).str.lower() != "nan"
    ].copy()
    
    print(len(command_hr_long))

    command_raw = trinoFetch(f"""
    SELECT
        iotDeviceNo AS deviceId,
        requestId,
        hookMeta
    FROM iceberg.bronze.battery_immob_remob_requests
    WHERE requestId IN ({request_ids_sql})
    """)

    command_raw = (
        command_raw
        .sort_values("requestId")
        .drop_duplicates(
            subset=["requestId", "deviceId"],
            keep="last"
        )
    )

    print(len(command_raw))

    command_raw["requestId"] = command_raw["requestId"].astype(str)
    command_hr_long["requestId"] = command_hr_long["requestId"].astype(str)

    command_check = command_hr_long.merge(
        command_raw[["requestId", "deviceId", "hookMeta"]],
        on=["requestId", "deviceId"],
        how="left"
    )

    command_check

    def get_success_ts(hook_meta):

        try:
            data = json.loads(hook_meta)

            statuses = (
                data.get("data", {})
                    .get("status", [])
            )

            for s in statuses:

                if (
                    s.get("cmdstatus")
                    == "DEVICE_CMD_EXECUTION_SUCCESS"
                ):
                    return s.get("timestamp")

            return np.nan

        except:
            return np.nan


    command_check["success_ts"] = command_check["hookMeta"].apply(get_success_ts)

    completed_hr_fail_cmd = command_check[command_check["success_ts"].isna()]
    print(len(completed_hr_fail_cmd))

    command_check["success_time"] = (
        pd.to_datetime(
            command_check["success_ts"],
            unit="ms"
        )
        + pd.Timedelta(minutes=330)
    )

    completed_hr_success_cmd=command_check[command_check["success_ts"].notna()]
    print(len(completed_hr_success_cmd))

    device_list_sql = ",".join(
        f"'{x}'"
        for x in completed_hr_success_cmd["deviceId"].dropna().unique()
    )

    all_pings=trinoFetch(f""" select iotdeviceno as deviceId,state, ts from iceberg.gold.iot_latest_state_logs where iotDeviceNo in ({device_list_sql}) and state in ('Charging','Discharging') and lat<>25 and lon<>-71 """)

    print(len(all_pings))

    all_pings["ts"] = pd.to_datetime(
        all_pings["ts"],
        errors="coerce"
    )

    completed_hr_success_cmd["success_ts"] = pd.to_datetime(
        completed_hr_success_cmd["success_ts"],
        errors="coerce"
    )

    completed_hr_success_cmd

    def check_expected_state_count(row):

        device_id = row["deviceId"]
        success_ts = row["success_time"]

        expected_state = (
            "Charging"
            if row["command_type"] == "CHG"
            else "Discharging"
        )

        temp = all_pings[
            (all_pings["deviceId"] == device_id)
            &
            (all_pings["ts"] >= success_ts)
        ]

        expected_count = (
            temp["state"]
            .astype(str)
            .str.lower()
            .eq(expected_state.lower())
            .sum()
        )

        return expected_count >= 3


    all_pings
    all_pings["ts"]=pd.to_datetime(all_pings["ts"]).dt.tz_localize(None)
    completed_hr_success_cmd["found_expected_state"] = (
        completed_hr_success_cmd.apply(
            check_expected_state_count,
            axis=1
        )
    )

    final_pass=completed_hr_success_cmd[completed_hr_success_cmd["found_expected_state"]==False]
    print(len(final_pass))

    custom_append("https://docs.google.com/spreadsheets/d/12kPyTZb1PILSulIN6uITJwy3uCvg-c8TJtgFd9UEZmw/edit?gid=1455408464#gid=1455408464","Pass_cmd",final_pass)

    final_fail=completed_hr_success_cmd[completed_hr_success_cmd["found_expected_state"]==True]
    print(len(final_fail))

    batteries_map=prodFetch('''SELECT 
        id AS batteryId,
        serialNo,
        manufacturerName,
        iotDeviceNo as deviceId,batteryType,phase
    FROM batteries
    WHERE deletedAt IS NULL
    AND isBaas = 0;''')

    phase_map = batteries_map[
        ["deviceId", "phase","batteryType","serialNo","manufacturerName"]
    ].drop_duplicates()

    final_fail = final_fail.merge(
        phase_map,
        on="deviceId",
        how="left"
    )

    completed_hr_fail_cmd = completed_hr_fail_cmd.merge(
        phase_map,
        on="deviceId",
        how="left"
    )
    cols = remaining_batteries.columns



    final_combined = pd.concat(
        [
            remaining_batteries,
            final_fail.reindex(columns=cols),
            completed_hr_fail_cmd.reindex(columns=cols)
        ],
        ignore_index=True
    )

else:
    final_combined =   remaining_batteries.copy()

In [83]:
final_combined

,deviceId,total_events,valid_events,batteryId,serialNo,manufacturerName,batteryType,phase,lat,lon,latest_unauth_charging_ts
0,011BB223A7E28202,3,0,B932911,B112004450_125992,BATTRIXX,LFP,2,28.492344,77.058230,2026-06-28 11:35:00+00:00
1,011F527F17C287A2,2,0,B1014136,MB3AJ5EPBB00349,LIVGUARD,LFP,2,28.867230,77.122090,2026-06-27 13:55:00+00:00
2,015F0F4C65AAFBBC,1,0,B605385,G22-39732,STEFEN ELECTRIC,NMC,1,28.608490,77.071785,2026-06-28 15:50:00+00:00
3,01D6E5248519D915,1,0,B957604,A24-32167,STEFEN ELECTRIC,NMC,2,28.613647,77.346670,2026-06-27 21:30:00+00:00
4,01F4BA65BA0B486A,5,0,B509265,B112003240_121904,BATTRIXX,LFP,2,26.804497,75.831245,2026-06-28 00:25:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...
1094,FDF791D41DE9BCC8,5,0,B858615,B112004450_124425,BATTRIXX,LFP,2,27.486206,77.669080,2026-06-28 00:25:00+00:00
1095,FE1D4B60492570EA,3,0,B340023,MD0AI2LKAA00033,LIVGUARD,NMC,1,28.656700,77.215840,2026-06-28 13:40:00+00:00
1096,FE81F968F4696636,4,0,B933011,B112004450_125979,BATTRIXX,LFP,2,26.154633,80.495350,2026-06-27 20:15:00+00:00
1097,FEB2099A86208B84,1,0,B940174,E23-10776,STEFEN ELECTRIC,NMC,1,26.927227,75.843690,2026-06-27 09:40:00+00:00


In [84]:
print(len(final_combined))

1096


In [85]:
final_combined = final_combined.drop_duplicates(
    subset=["deviceId"],
    keep="first"
).reset_index(drop=True)

print(len(final_combined))

1094


In [86]:
print(len(final_combined))

1094


In [87]:
SEED   = 42
strata = ['manufacturerName', 'batteryType', 'phase']

fbm  = final_combined.sample(frac=1, random_state=SEED)      # shuffle for random within-stratum assignment
grp  = fbm.groupby(strata, dropna=False)                        # dropna=False -> NaN strata still assigned, nothing dropped
rank = grp.cumcount()
n    = grp['batteryId'].transform('size')
fbm['mosfet_action'] = np.where(rank < np.ceil(n / 2), 'discharging', 'charging')   # ceil -> discharging gets the extra
final = fbm.sort_index()
final

,deviceId,total_events,valid_events,batteryId,serialNo,manufacturerName,batteryType,phase,lat,lon,latest_unauth_charging_ts,mosfet_action
0,011BB223A7E28202,3,0,B932911,B112004450_125992,BATTRIXX,LFP,2,28.492344,77.058230,2026-06-28 11:35:00+00:00,discharging
1,011F527F17C287A2,2,0,B1014136,MB3AJ5EPBB00349,LIVGUARD,LFP,2,28.867230,77.122090,2026-06-27 13:55:00+00:00,charging
2,015F0F4C65AAFBBC,1,0,B605385,G22-39732,STEFEN ELECTRIC,NMC,1,28.608490,77.071785,2026-06-28 15:50:00+00:00,discharging
3,01D6E5248519D915,1,0,B957604,A24-32167,STEFEN ELECTRIC,NMC,2,28.613647,77.346670,2026-06-27 21:30:00+00:00,discharging
4,01F4BA65BA0B486A,5,0,B509265,B112003240_121904,BATTRIXX,LFP,2,26.804497,75.831245,2026-06-28 00:25:00+00:00,charging
...,...,...,...,...,...,...,...,...,...,...,...,...
1089,FDF791D41DE9BCC8,5,0,B858615,B112004450_124425,BATTRIXX,LFP,2,27.486206,77.669080,2026-06-28 00:25:00+00:00,charging
1090,FE1D4B60492570EA,3,0,B340023,MD0AI2LKAA00033,LIVGUARD,NMC,1,28.656700,77.215840,2026-06-28 13:40:00+00:00,discharging
1091,FE81F968F4696636,4,0,B933011,B112004450_125979,BATTRIXX,LFP,2,26.154633,80.495350,2026-06-27 20:15:00+00:00,discharging
1092,FEB2099A86208B84,1,0,B940174,E23-10776,STEFEN ELECTRIC,NMC,1,26.927227,75.843690,2026-06-27 09:40:00+00:00,charging


In [88]:
final.drop(
    columns=["total_events", "valid_events"],
    inplace=True
)

In [89]:
final['mosfet_action'].value_counts()

mosfet_action
discharging    551
charging       543
Name: count, dtype: int64

In [90]:
final

,deviceId,batteryId,serialNo,manufacturerName,batteryType,phase,lat,lon,latest_unauth_charging_ts,mosfet_action
0,011BB223A7E28202,B932911,B112004450_125992,BATTRIXX,LFP,2,28.492344,77.058230,2026-06-28 11:35:00+00:00,discharging
1,011F527F17C287A2,B1014136,MB3AJ5EPBB00349,LIVGUARD,LFP,2,28.867230,77.122090,2026-06-27 13:55:00+00:00,charging
2,015F0F4C65AAFBBC,B605385,G22-39732,STEFEN ELECTRIC,NMC,1,28.608490,77.071785,2026-06-28 15:50:00+00:00,discharging
3,01D6E5248519D915,B957604,A24-32167,STEFEN ELECTRIC,NMC,2,28.613647,77.346670,2026-06-27 21:30:00+00:00,discharging
4,01F4BA65BA0B486A,B509265,B112003240_121904,BATTRIXX,LFP,2,26.804497,75.831245,2026-06-28 00:25:00+00:00,charging
...,...,...,...,...,...,...,...,...,...,...
1089,FDF791D41DE9BCC8,B858615,B112004450_124425,BATTRIXX,LFP,2,27.486206,77.669080,2026-06-28 00:25:00+00:00,charging
1090,FE1D4B60492570EA,B340023,MD0AI2LKAA00033,LIVGUARD,NMC,1,28.656700,77.215840,2026-06-28 13:40:00+00:00,discharging
1091,FE81F968F4696636,B933011,B112004450_125979,BATTRIXX,LFP,2,26.154633,80.495350,2026-06-27 20:15:00+00:00,discharging
1092,FEB2099A86208B84,B940174,E23-10776,STEFEN ELECTRIC,NMC,1,26.927227,75.843690,2026-06-27 09:40:00+00:00,charging


In [91]:
no_sleep = trinoFetch("""
SELECT iotdeviceno as deviceId, state, ts
FROM iceberg.gold.iot_latest_state
WHERE state IN ('Charging', 'Discharging')
""")

In [92]:
final_no_sleep=final[final["deviceId"].isin(no_sleep["deviceId"])].copy()

In [93]:
final_no_sleep = (
    final
    .merge(
        no_sleep[
            ["deviceId", "state"]
        ],
        on="deviceId",
        how="inner"
    )
)

In [94]:
final_no_sleep["final_timestamp"] = (
    datetime.now(
        pytz.timezone("Asia/Kolkata")
    ).strftime(
        "%Y-%m-%d %H:%M:%S.%f"
    ) + "+00:00"
)

In [95]:
print(len(final_no_sleep))

267


In [96]:
append_logs = []
# raw_logs=[]

In [97]:
ist_time = datetime.now(
    pytz.timezone("Asia/Kolkata")
)

In [99]:
final_no_sleep

,deviceId,batteryId,serialNo,manufacturerName,batteryType,phase,lat,lon,latest_unauth_charging_ts,mosfet_action,state,final_timestamp
0,011BB223A7E28202,B932911,B112004450_125992,BATTRIXX,LFP,2,28.492344,77.058230,2026-06-28 11:35:00+00:00,discharging,Discharging,2026-06-29 11:43:31.974409+00:00
1,025EC683490668B4,B617110,INDMWE514000961,INVERTED,NMC,1,28.412035,76.989586,2026-06-27 19:20:00+00:00,discharging,Charging,2026-06-29 11:43:31.974409+00:00
2,03949429369F365B,B651605,B112004180_127718,BATTRIXX,LFP,2,26.875858,80.922680,2026-06-26 00:00:00+00:00,charging,Discharging,2026-06-29 11:43:31.974409+00:00
3,04C35E22578CD46C,B509015,B112003240_121622,BATTRIXX,LFP,2,26.961569,75.870640,2026-06-27 15:00:00+00:00,charging,Charging,2026-06-29 11:43:31.974409+00:00
4,081A2F4A3AEE2347,B908671,INCBZA5145E0775,INVERTED,LFP,2,26.875630,80.922960,2026-06-27 11:25:00+00:00,charging,Charging,2026-06-29 11:43:31.974409+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
262,FA5D906A1BEB81A6,B816297,B112004450_121097,BATTRIXX,LFP,2,26.961517,75.870670,2026-06-28 10:35:00+00:00,charging,Charging,2026-06-29 11:43:31.974409+00:00
263,FA98E2293972BCB1,B506258,INDMXL5140E0425,INVERTED,NMC,2,26.466970,80.370224,2026-06-25 18:55:00+00:00,charging,Discharging,2026-06-29 11:43:31.974409+00:00
264,FBCE994B33638882,B388344,I23-21561,STEFEN ELECTRIC,NMC,1,28.530247,77.319214,2026-06-26 23:20:00+00:00,discharging,Discharging,2026-06-29 11:43:31.974409+00:00
265,FD0B02488A6BD631,B676891,INDMWJ514002337,INVERTED,NMC,1,26.466957,80.370140,2026-06-28 11:30:00+00:00,charging,Charging,2026-06-29 11:43:31.974409+00:00


In [100]:
for idx, row in final_no_sleep.iterrows():

    battery_id = row["batteryId"]
    serial_no = row["serialNo"]
    device_id = row["deviceId"]
    manufacturer_name = row["manufacturerName"]
    lat =row["lat"]
    lon = row["lon"]
    state_now =row["state"]
    mosfet_action = str(row["mosfet_action"]).lower()
    

    current_time = datetime.now(
        pytz.timezone("Asia/Kolkata")
    ).strftime(
        "%Y-%m-%d %H:%M:%S.%f"
    ) + "+00:00"

    # =====================================
    # COMMAND SELECTION
    # =====================================

    if mosfet_action == "charging":

        command = "CHGOFF"

    elif mosfet_action == "discharging":

        command = "DISCHGOFF"

    else:

        print(
            f"\nSKIPPING -> {battery_id}"
        )
        print(
            f"INVALID mosfet_action -> {mosfet_action}"
        )
        continue

    print(f"\nPROCESSING -> {battery_id}")
    print(f"  ACTION -> {mosfet_action}")
    print(f"  COMMAND -> {command}")

    # =====================================
    # SEND COMMAND
    # =====================================

    resp = send_command(
        serial_no,
        command
    )



    success = resp["success"]

    # =====================================
    # SUCCESS
    # =====================================

    if success:

        append_logs.append({

            "batteryId": battery_id,
            "serialNo": serial_no,
            "deviceId": device_id,
            "manufacturerName": manufacturer_name,
            "lat":lat,
            "lon":lon,
            "action_timestamp": current_time,
            "state": state_now,
            "action": "turned_off",
            

            "CHG CMD": (
                "CHGOFF"
                if command == "CHGOFF"
                else np.nan
            ),

            "DICHG CMD": (
                "DISCHGOFF"
                if command == "DISCHGOFF"
                else np.nan
            ),

            "chg_rid": (
                resp["request_id"]
                if command == "CHGOFF"
                else np.nan
            ),

            "dischg_rid": (
                resp["request_id"]
                if command == "DISCHGOFF"
                else np.nan
            )
        })

        print(
            "  COMMAND SUCCESS"
        )

    # =====================================
    # FAILURE
    # =====================================

    else:

        raw_logs.append({

            "batteryId": battery_id,
            "serialNo": serial_no,
            "deviceId": device_id,
            "manufacturerName": manufacturer_name,
            "lat":lat,
            "lon":lon,
            "action_timestamp": current_time,
            "state": state_now,
            "action": "turn_off_failed",
            
            

            "CHG CMD": (
                "CHGON"
                if command == "CHGOFF"
                else np.nan
            ),

            "DICHG CMD": (
                "DISCHGON"
                if command == "DISCHGOFF"
                else np.nan
            ),

            "chg_rid": (
                resp["request_id"]
                if command == "CHGOFF"
                else np.nan
            ),

            "dischg_rid": (
                resp["request_id"]
                if command == "DISCHGOFF"
                else np.nan
            )
        })

        print(
            "  COMMAND FAILED"
        )

# =====================================
# DATAFRAMES
# =====================================
append_logs = pd.DataFrame(append_logs)
raw_logs = pd.DataFrame(raw_logs)



PROCESSING -> B932911
  ACTION -> discharging
  COMMAND -> DISCHGOFF
  COMMAND SUCCESS

PROCESSING -> B617110
  ACTION -> discharging
  COMMAND -> DISCHGOFF
  COMMAND SUCCESS

PROCESSING -> B651605
  ACTION -> charging
  COMMAND -> CHGOFF
  COMMAND SUCCESS

PROCESSING -> B509015
  ACTION -> charging
  COMMAND -> CHGOFF
  COMMAND SUCCESS

PROCESSING -> B908671
  ACTION -> charging
  COMMAND -> CHGOFF
  COMMAND SUCCESS

PROCESSING -> B712195
  ACTION -> discharging
  COMMAND -> DISCHGOFF
  COMMAND SUCCESS

PROCESSING -> B1027016
  ACTION -> charging
  COMMAND -> CHGOFF
  COMMAND SUCCESS

PROCESSING -> B496566
  ACTION -> discharging
  COMMAND -> DISCHGOFF
  COMMAND SUCCESS

PROCESSING -> B953477
  ACTION -> charging
  COMMAND -> CHGOFF
  COMMAND SUCCESS

PROCESSING -> B914759
  ACTION -> discharging
  COMMAND -> DISCHGOFF
  COMMAND SUCCESS

PROCESSING -> B940274
  ACTION -> discharging
  COMMAND -> DISCHGOFF
  COMMAND SUCCESS

PROCESSING -> B1001115
  ACTION -> charging
  COMMAND -> CHG

In [101]:
print("\nSUCCESS COUNT :", len(append_logs))
print("FAIL COUNT    :", len(raw_logs))


SUCCESS COUNT : 256
FAIL COUNT    : 11


In [102]:
tracking_df

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
0,B605385,015F0F4C65AAFBBC,STEFEN ELECTRIC,G22-39732,2026-05-21 11:29:43.289487,2026-05-21,,turned_off,,,,,,
1,B399907,D928BD6505A2A461,INVERTED,INDMXD5140E0048,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
2,B405455,C5FBD9AE73EC1B17,LIVGUARD,MD4AIGDNAB06921,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
3,B419006,AC37695380E1959D,LIVGUARD,MD4AIVENBB03819,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
4,B427128,F2453474865D6473,LIVGUARD,MD4AI9GNBB01474,2026-05-21 17:51:43.289487,2026-05-21,,turned_off,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11775,B769624,30522C43A6157217,INVERTED,INCBYH5145E1320,2026-06-26 16:10:40.896202,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,461293_8,,
11776,B902526,2D203487FE1C6F8A,BATTRIXX,B112003240_121494,2026-06-26 16:10:41.912223,2026-06-26,Discharging,turned_off,NaN,DISCHGOFF,NaN,262006_5,,
11777,B607994,605F0EDEE94D048B,STEFEN ELECTRIC,B24-33490,2026-06-26 16:10:53.401240,2026-06-26,Discharging,turned_off,CHGOFF,NaN,169610_25,NaN,,
11778,B709679,331FC418E7902E2E,IPOWER,HHL48452507011864,2026-06-26 16:10:54.414728,2026-06-26,Charging,turned_off,CHGOFF,NaN,446417_3,NaN,,


In [103]:
final_no_sleep[final_no_sleep['serialNo']=='B112004420_122193']

,deviceId,batteryId,serialNo,manufacturerName,batteryType,phase,lat,lon,latest_unauth_charging_ts,mosfet_action,state,final_timestamp
76,53C0E4484535AEC6,B696830,B112004420_122193,BATTRIXX,LFP,2,29.466759,77.70318,2026-06-27 16:25:00+00:00,discharging,Charging,2026-06-29 11:43:31.974409+00:00


In [104]:
append_logs_df = pd.DataFrame(
    append_logs
)

In [105]:
append_logs_df.tail(40)

,batteryId,serialNo,deviceId,manufacturerName,lat,lon,action_timestamp,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid
216,B658525,MD2AIFFLAA00366,DE9FB6BEE79BE3D0,LIVGUARD,26.956175,75.837975,2026-06-29 11:49:01.754374+00:00,Discharging,turned_off,CHGOFF,NaN,162847_5,NaN
217,B843866,J24-60945,E07581058F2453C1,STEFEN ELECTRIC,28.502829,77.250145,2026-06-29 11:49:02.965582+00:00,Charging,turned_off,NaN,DISCHGOFF,NaN,281535_1
218,B413614,A24-32212,E0B9DE7162E4D23C,STEFEN ELECTRIC,28.981647,77.689545,2026-06-29 11:49:04.118329+00:00,Discharging,turned_off,CHGOFF,NaN,176030_2,NaN
219,B512362,INDMXK5140E0840,E1998F57F9DEC9D6,INVERTED,26.961555,75.870605,2026-06-29 11:49:05.343029+00:00,Charging,turned_off,CHGOFF,NaN,267584_1,NaN
220,B649074,B112004180_121108,E1D152F7EEFB6E4D,BATTRIXX,26.923176,75.848015,2026-06-29 11:49:06.507854+00:00,Discharging,turned_off,CHGOFF,NaN,338028_12,NaN
221,B689792,B112004180_129904,E1E85D297122B11C,BATTRIXX,26.933530,75.807750,2026-06-29 11:49:07.627299+00:00,Discharging,turned_off,CHGOFF,NaN,346216_2,NaN
222,B650052,B112004180_125360,E354089FC51E571F,BATTRIXX,25.305138,83.047874,2026-06-29 11:49:09.444281+00:00,Discharging,turned_off,CHGOFF,NaN,359981_6,NaN
223,B479029,INDMWI514002403,E5B37DB529E28B0C,INVERTED,28.678976,77.061104,2026-06-29 11:49:10.594337+00:00,Charging,turned_off,CHGOFF,NaN,161448_38,NaN
224,B427816,MD4AIQFNAB03045,E5F2E5C2A0D70A54,LIVGUARD,29.094528,77.264790,2026-06-29 11:49:12.041066+00:00,Discharging,turned_off,NaN,DISCHGOFF,NaN,221966_1
225,B800622,HHL48452508016058,E622216910379748,IPOWER,26.933418,75.807700,2026-06-29 11:49:13.439407+00:00,Discharging,turned_off,NaN,DISCHGOFF,NaN,474705_1


In [106]:
append_logs_df["action_timestamp"] = pd.to_datetime(
    append_logs_df["action_timestamp"]
)

append_logs_df["action_date"] = (
    append_logs_df["action_timestamp"]
    .dt.date
)

In [107]:
append_logs_df.head()

,batteryId,serialNo,deviceId,manufacturerName,lat,lon,action_timestamp,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,action_date
0,B932911,B112004450_125992,011BB223A7E28202,BATTRIXX,28.492344,77.058230,2026-06-29 11:44:27.118489+00:00,Discharging,turned_off,NaN,DISCHGOFF,NaN,490684_6,2026-06-29
1,B617110,INDMWE514000961,025EC683490668B4,INVERTED,28.412035,76.989586,2026-06-29 11:44:28.403005+00:00,Charging,turned_off,NaN,DISCHGOFF,NaN,128747_14,2026-06-29
2,B651605,B112004180_127718,03949429369F365B,BATTRIXX,26.875858,80.922680,2026-06-29 11:44:29.536374+00:00,Discharging,turned_off,CHGOFF,NaN,360466_3,NaN,2026-06-29
3,B509015,B112003240_121622,04C35E22578CD46C,BATTRIXX,26.961569,75.870640,2026-06-29 11:44:30.617170+00:00,Charging,turned_off,CHGOFF,NaN,262078_8,NaN,2026-06-29
4,B908671,INCBZA5145E0775,081A2F4A3AEE2347,INVERTED,26.875630,80.922960,2026-06-29 11:44:31.754611+00:00,Charging,turned_off,CHGOFF,NaN,575172_1,NaN,2026-06-29


In [108]:
append_logs_df

,batteryId,serialNo,deviceId,manufacturerName,lat,lon,action_timestamp,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,action_date
0,B932911,B112004450_125992,011BB223A7E28202,BATTRIXX,28.492344,77.058230,2026-06-29 11:44:27.118489+00:00,Discharging,turned_off,NaN,DISCHGOFF,NaN,490684_6,2026-06-29
1,B617110,INDMWE514000961,025EC683490668B4,INVERTED,28.412035,76.989586,2026-06-29 11:44:28.403005+00:00,Charging,turned_off,NaN,DISCHGOFF,NaN,128747_14,2026-06-29
2,B651605,B112004180_127718,03949429369F365B,BATTRIXX,26.875858,80.922680,2026-06-29 11:44:29.536374+00:00,Discharging,turned_off,CHGOFF,NaN,360466_3,NaN,2026-06-29
3,B509015,B112003240_121622,04C35E22578CD46C,BATTRIXX,26.961569,75.870640,2026-06-29 11:44:30.617170+00:00,Charging,turned_off,CHGOFF,NaN,262078_8,NaN,2026-06-29
4,B908671,INCBZA5145E0775,081A2F4A3AEE2347,INVERTED,26.875630,80.922960,2026-06-29 11:44:31.754611+00:00,Charging,turned_off,CHGOFF,NaN,575172_1,NaN,2026-06-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,B557459,MD2AI8DLAA00254,FA15D986A45597DF,LIVGUARD,28.664091,77.180214,2026-06-29 11:49:57.138701+00:00,Discharging,turned_off,CHGOFF,NaN,103975_28,NaN,2026-06-29
252,B855240,INCBYK5145E0440,FA17DA7B5DB9B913,INVERTED,26.875874,80.922830,2026-06-29 11:49:58.341526+00:00,Discharging,turned_off,CHGOFF,NaN,527437_1,NaN,2026-06-29
253,B816297,B112004450_121097,FA5D906A1BEB81A6,BATTRIXX,26.961517,75.870670,2026-06-29 11:49:59.605893+00:00,Charging,turned_off,CHGOFF,NaN,475067_4,NaN,2026-06-29
254,B506258,INDMXL5140E0425,FA98E2293972BCB1,INVERTED,26.466970,80.370224,2026-06-29 11:50:00.782966+00:00,Discharging,turned_off,CHGOFF,NaN,298277_1,NaN,2026-06-29


In [109]:
append_logs_df=append_logs_df[["batteryId","deviceId","manufacturerName","serialNo","action_timestamp","action_date","state","action","CHG CMD","DICHG CMD","chg_rid","dischg_rid","lat","lon"]]

In [110]:
append_logs_df

,batteryId,deviceId,manufacturerName,serialNo,action_timestamp,action_date,state,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,lat,lon
0,B932911,011BB223A7E28202,BATTRIXX,B112004450_125992,2026-06-29 11:44:27.118489+00:00,2026-06-29,Discharging,turned_off,NaN,DISCHGOFF,NaN,490684_6,28.492344,77.058230
1,B617110,025EC683490668B4,INVERTED,INDMWE514000961,2026-06-29 11:44:28.403005+00:00,2026-06-29,Charging,turned_off,NaN,DISCHGOFF,NaN,128747_14,28.412035,76.989586
2,B651605,03949429369F365B,BATTRIXX,B112004180_127718,2026-06-29 11:44:29.536374+00:00,2026-06-29,Discharging,turned_off,CHGOFF,NaN,360466_3,NaN,26.875858,80.922680
3,B509015,04C35E22578CD46C,BATTRIXX,B112003240_121622,2026-06-29 11:44:30.617170+00:00,2026-06-29,Charging,turned_off,CHGOFF,NaN,262078_8,NaN,26.961569,75.870640
4,B908671,081A2F4A3AEE2347,INVERTED,INCBZA5145E0775,2026-06-29 11:44:31.754611+00:00,2026-06-29,Charging,turned_off,CHGOFF,NaN,575172_1,NaN,26.875630,80.922960
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251,B557459,FA15D986A45597DF,LIVGUARD,MD2AI8DLAA00254,2026-06-29 11:49:57.138701+00:00,2026-06-29,Discharging,turned_off,CHGOFF,NaN,103975_28,NaN,28.664091,77.180214
252,B855240,FA17DA7B5DB9B913,INVERTED,INCBYK5145E0440,2026-06-29 11:49:58.341526+00:00,2026-06-29,Discharging,turned_off,CHGOFF,NaN,527437_1,NaN,26.875874,80.922830
253,B816297,FA5D906A1BEB81A6,BATTRIXX,B112004450_121097,2026-06-29 11:49:59.605893+00:00,2026-06-29,Charging,turned_off,CHGOFF,NaN,475067_4,NaN,26.961517,75.870670
254,B506258,FA98E2293972BCB1,INVERTED,INDMXL5140E0425,2026-06-29 11:50:00.782966+00:00,2026-06-29,Discharging,turned_off,CHGOFF,NaN,298277_1,NaN,26.466970,80.370224


In [111]:
custom_append(
        "https://docs.google.com/spreadsheets/d/12kPyTZb1PILSulIN6uITJwy3uCvg-c8TJtgFd9UEZmw/edit?gid=1581823282#gid=1581823282",
        "logs",append_logs_df
    )

Appended successfully.


In [112]:
raw_logs_df=pd.DataFrame(raw_logs)
raw_logs_df['date'] = pd.to_datetime(raw_logs_df['action_timestamp']).dt.date
raw_logs_df=raw_logs_df[['batteryId', 'serialNo', 'deviceId', 'manufacturerName',
       'action_timestamp',  'action', 'CHG CMD', 'DICHG CMD',
       'chg_rid', 'dischg_rid','state','date','lat','lon']]

In [113]:
raw_logs_df

,batteryId,serialNo,deviceId,manufacturerName,action_timestamp,action,CHG CMD,DICHG CMD,chg_rid,dischg_rid,state,date,lat,lon
0,B729046,E23-10708,3476D0D8E62BB1F2,STEFEN ELECTRIC,2026-06-29 11:45:20.260787+00:00,turn_off_failed,NaN,DISCHGON,NaN,NaN,Discharging,2026-06-29,26.927197,75.843670
1,B629504,INDMWI514001033,3F2F55F6F0F1B16E,INVERTED,2026-06-29 11:45:30.426733+00:00,turn_off_failed,NaN,DISCHGON,NaN,NaN,Discharging,2026-06-29,27.568214,81.623490
2,B485455,E2W5140BS3724624,488CCD9C45AAEA1D,CYGNI ENERGY,2026-06-29 11:45:45.113344+00:00,turn_off_failed,NaN,DISCHGON,NaN,NaN,Discharging,2026-06-29,28.620724,77.368660
3,B952986,INDMWG514002147,594726AA3B8C63F0,INVERTED,2026-06-29 11:46:07.887526+00:00,turn_off_failed,NaN,DISCHGON,NaN,NaN,Charging,2026-06-29,28.655891,77.185250
4,B486899,E2W5140BS4024883,7237A6D3C144ACF2,CYGNI ENERGY,2026-06-29 11:46:34.964057+00:00,turn_off_failed,NaN,DISCHGON,NaN,NaN,Discharging,2026-06-29,28.663235,77.180700
5,B598593,INDMWH514000453,79B5D9EAE21707FE,INVERTED,2026-06-29 11:46:51.483648+00:00,turn_off_failed,CHGON,NaN,NaN,NaN,Discharging,2026-06-29,29.284828,77.943330
6,B673966,INDMWG514002511,7C900FD2E3368E17,INVERTED,2026-06-29 11:46:53.186529+00:00,turn_off_failed,NaN,DISCHGON,NaN,NaN,Discharging,2026-06-29,28.621376,77.329450
7,B409899,INDMWJ514002679,90A5BB446BD93F87,INVERTED,2026-06-29 11:47:23.175669+00:00,turn_off_failed,CHGON,NaN,NaN,NaN,Discharging,2026-06-29,25.676146,83.503100
8,B438212,INDMXF5140E1610,E2CE9E84D239303E,INVERTED,2026-06-29 11:49:08.858792+00:00,turn_off_failed,CHGON,NaN,NaN,NaN,Discharging,2026-06-29,25.606710,83.863380
9,B388344,I23-21561,FBCE994B33638882,STEFEN ELECTRIC,2026-06-29 11:50:01.883631+00:00,turn_off_failed,NaN,DISCHGON,NaN,NaN,Discharging,2026-06-29,28.530247,77.319214


In [114]:
custom_append(
        "https://docs.google.com/spreadsheets/d/12kPyTZb1PILSulIN6uITJwy3uCvg-c8TJtgFd9UEZmw/edit?gid=1581823282#gid=1581823282",
        "raw_logs",raw_logs_df
    )

Appended successfully.
